In [1]:
%%writefile /tmp/gen_fe.py
GROUPS = 150          # barrier+add blocks -> ~1.9 KiB... see cell output
REGS = "abcdefgh"

lines = []
for g in range(GROUPS):
    lines.append("    " + " ".join(f'__asm__("" : "+r"({r}));' for r in REGS))
    lines.append("    " + " ".join(f"{r} += {g * len(REGS) + k + 1};" for k, r in enumerate(REGS)))

source = f"""
extern "C" long fe_code(long a, long b, long c, long d) {{
    long e = a + 1, f = b + 1, g = c + 1, h = d + 1;
{chr(10).join(lines)}
    return a + b + c + d + e + f + g + h;
}}
"""

with open("/tmp/study_fe.cpp", "w") as fh:
    fh.write(source)

Overwriting /tmp/gen_fe.py


In [2]:
!python /tmp/gen_fe.py && cat /tmp/study_fe.cpp


extern "C" long fe_code(long a, long b, long c, long d) {
    long e = a + 1, f = b + 1, g = c + 1, h = d + 1;
    __asm__("" : "+r"(a)); __asm__("" : "+r"(b)); __asm__("" : "+r"(c)); __asm__("" : "+r"(d)); __asm__("" : "+r"(e)); __asm__("" : "+r"(f)); __asm__("" : "+r"(g)); __asm__("" : "+r"(h));
    a += 1; b += 2; c += 3; d += 4; e += 5; f += 6; g += 7; h += 8;
    __asm__("" : "+r"(a)); __asm__("" : "+r"(b)); __asm__("" : "+r"(c)); __asm__("" : "+r"(d)); __asm__("" : "+r"(e)); __asm__("" : "+r"(f)); __asm__("" : "+r"(g)); __asm__("" : "+r"(h));
    a += 9; b += 10; c += 11; d += 12; e += 13; f += 14; g += 15; h += 16;
    __asm__("" : "+r"(a)); __asm__("" : "+r"(b)); __asm__("" : "+r"(c)); __asm__("" : "+r"(d)); __asm__("" : "+r"(e)); __asm__("" : "+r"(f)); __asm__("" : "+r"(g)); __asm__("" : "+r"(h));
    a += 17; b += 18; c += 19; d += 20; e += 21; f += 22; g += 23; h += 24;
    __asm__("" : "+r"(a)); __asm__("" : "+r"(b)); __asm__("" : "+r"(c)); __asm__("" : "+r"(d)); __asm__("

In [3]:
!g++ -O3 -c /tmp/study_fe.cpp -o /tmp/study_fe.o

In [4]:
import subprocess

text = subprocess.run(
    ["objdump", "-d", "--no-show-raw-insn", "/tmp/study_fe.o"],
    capture_output=True, text=True, check=True,
).stdout
body = text.split("<fe_code>:", 1)[1]
insns = [l for l in body.splitlines() if l.strip() and l.strip()[0].isdigit()]
nbytes = int(insns[-1].split(":")[0].strip(), 16)
print(f"{len(insns)} instructions, {nbytes} bytes = {nbytes / 4096:.1f} pages of code")

959 instructions, 7912 bytes = 1.9 pages of code


## Frontend bound: hot vs cold instruction cache

One axis, one function, two tiers. `fe_code` is a large straight-line body that
does **eight independent adds per group** across 150 groups.

Two properties matter, and both are required:

- **High ILP.** Eight independent accumulator chains mean the backend is never
  the limiter, so there is slack for a front-end stall to show up as cycles.
  An earlier version of this notebook used one dependent accumulator. It showed a
  `fe-bound` share of 0.129 but *identical* cycle counts hot and cold — the
  backend simply absorbed the misses, and the axis was not measurable.
- **A footprint that does not fit L1i.** ~1200 instructions over ~2 pages is well
  past what stays resident, so evicting the code has something to cost.

**Claim.** With `icache` set to `cold`, the function retires the same
instructions in far more cycles, and the bound moves: `topdown-fe-bound` becomes
the dominant slot, where with `hot` the dominant slot is `topdown-retiring`.

The tiers are not a simulation: perf evicts code pages with `mprotect`, which
takes the lines out of L1i and the uop cache through coherence, so `cold` pays a
real re-fetch on the next pass.

In [5]:
import pandas as pd

import perf

TARGET = "/tmp/study_fe.o"
FUNC = "fe_code"
DATA = {"regs": {"rdi": 7, "rsi": 8, "rdx": 9, "rcx": 10}}

# Every axis except the one under study is pinned, so the sweep differs only in
# `icache`. `icache` is one of the four cache/TLB axes perf can drive; the tiers
# below are the ones that matter here.
SLOTS = [
    "topdown-retiring",
    "topdown-bad-spec",
    "topdown-fe-bound",
    "topdown-be-bound",
]
RAW = ["cycles", "instructions"]
TIERS = ["hot", "cold"]

CONFIG = {
    "samples": 25,
    "iterations": 50,
    "branch": "predictable",
    "dcache": "hot",
    "dtlb": "hot",
    "code": {"align": 16},
    "func": {"align": 16, "order": "as-is"},
    "stack": {"size": 0x200000, "align": 16},
    "backend": {"loop": {"runs": 4}},
}


def sweep(event):
    """Benchmark one event set across the `icache` tiers."""
    frames = []
    for tier in TIERS:
        df = perf.benchmark(
            code=[TARGET, FUNC],
            mode=["latency"],
            event=event,
            config={**CONFIG, "icache": tier},
            data=DATA,
        ).reset_index()
        df["tier"] = tier
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def medians(df, columns):
    return df.groupby("tier")[columns].median()


def pct(med):
    return med.div(med.sum(axis=1), axis=0).round(3)


def check(claim, ok):
    print(("PASS  " if ok else "FAIL  ") + claim)
    assert ok, claim

In [6]:
TOP = medians(sweep(SLOTS), SLOTS)
print(TOP.round(1))
print()
print(pct(TOP))

CMP = medians(sweep(RAW), RAW)
CMP["ipc"] = CMP["instructions"] / CMP["cycles"]
print()
print(CMP.round(2))

share = pct(TOP)
hot_cyc = CMP.loc["hot", "cycles"]
cold_cyc = CMP.loc["cold", "cycles"]
hot_insn = CMP.loc["hot", "instructions"]
cold_insn = CMP.loc["cold", "instructions"]

check(
    f"both tiers retire the same instructions "
    f"({hot_insn:.0f} hot vs {cold_insn:.0f} cold)",
    0.99 <= cold_insn / hot_insn <= 1.01,
)
check(
    f"cold costs more cycles ({cold_cyc:.0f} > {hot_cyc:.0f}, "
    f"{cold_cyc / hot_cyc:.2f}x)",
    cold_cyc > 2 * hot_cyc,
)
check(
    f"cold is frontend bound: fe-bound "
    f"{share.loc['cold', 'topdown-fe-bound']} > 0.5 and is the largest slot",
    share.loc["cold", "topdown-fe-bound"] > 0.5
    and share.loc["cold"].idxmax() == "topdown-fe-bound",
)
check(
    f"hot is not: retiring "
    f"{share.loc['hot', 'topdown-retiring']} > 0.5 and is the largest slot",
    share.loc["hot", "topdown-retiring"] > 0.5
    and share.loc["hot"].idxmax() == "topdown-retiring",
)
check(
    f"backend-bound stays small either way "
    f"({share.loc['hot', 'topdown-be-bound']} hot, "
    f"{share.loc['cold', 'topdown-be-bound']} cold) so this is a frontend axis",
    share["topdown-be-bound"].max() < 0.1,
)

      topdown-retiring  topdown-bad-spec  topdown-fe-bound  topdown-be-bound
tier                                                                        
cold            1209.0               5.0            3813.0              48.0
hot             1209.0              20.0             373.0              28.0

      topdown-retiring  topdown-bad-spec  topdown-fe-bound  topdown-be-bound
tier                                                                        
cold             0.238             0.001             0.751             0.009
hot              0.742             0.012             0.229             0.017



      cycles  instructions   ipc
tier                            
cold   783.5        1209.0  1.54
hot    250.0        1209.0  4.84
PASS  both tiers retire the same instructions (1209 hot vs 1209 cold)
PASS  cold costs more cycles (784 > 250, 3.13x)
PASS  cold is frontend bound: fe-bound 0.751 > 0.5 and is the largest slot
PASS  hot is not: retiring 0.742 > 0.5 and is the largest slot
PASS  backend-bound stays small either way (0.017 hot, 0.009 cold) so this is a frontend axis


In [7]:
perf.plot(TOP, "boxen", event="topdown-fe-bound", groupby="tier")


P0;0;0q"1;1;637;485#0;2;0;0;0#1;2;4;4;4#2;2;7;7;7#3;2;11;11;11#4;2;14;14;14#5;2;17;17;17#6;2;21;21;21#7;2;24;24;24#8;2;27;27;27#9;2;29;29;27#10;2;27;28;29#11;2;30;30;30#12;2;33;33;33#13;2;36;36;36#14;2;39;39;39#15;2;42;42;42#16;2;45;45;45#17;2;49;49;49#18;2;74;52;13#19;2;75;53;13#20;2;76;53;13#21;2;9;42;61#22;2;9;42;60#23;2;51;51;51#24;2;55;55;55#25;2;58;58;58#26;2;61;61;61#27;2;64;64;64#28;2;67;67;67#29;2;70;70;70#30;2;74;74;74#31;2;77;77;77#32;2;80;80;80#33;2;83;83;83#34;2;86;86;86#35;2;89;89;89#36;2;93;93;93#37;2;95;95;95#38;2;100;100;100#38!637~-!637~-!564~^N!6^N^^N^^^N^N^^^N^!22~FFF!6~FFF!16~$#10!564?_#6!4_?!13_??_#10_#37!22?G?G!6?G?G$!565?O!6?O??O???O?O???O#33!23?O#0o!8?o#34o$#7!569?_!13?__#34!24?_#24G!8?G$#35!611?o!6?o-#38!563~T#6~#21u!10~}}!7~i#7b!6?_!7?G#25?@#34?G???G!8?O?~#37@$!563?i#22?H!10?@@!7?T#6[!6?G#1A!7?A??A#30A!6?C?@?@#6A$#38!587?W!4~B@w{{wx~B@w{{?@B~???~B@w{{???}!15~$#37!587?f#15!4?G!7?_A???C!7?g#13Q?A#7A#31g$#11!592?O??A#36?D!7?@#4C#29O#12!5?O!4?C$#14!592?_#0C!7?C

[]

In [8]:
perf.plot(CMP, "boxen", event="cycles", groupby="tier")


P0;0;0q"1;1;637;485#0;2;0;0;0#1;2;4;4;4#2;2;8;8;8#3;2;11;11;11#4;2;14;14;14#5;2;17;17;17#6;2;21;21;21#7;2;23;23;23#8;2;26;26;26#9;2;29;29;27#10;2;27;28;29#11;2;30;30;30#12;2;33;33;33#13;2;36;36;36#14;2;39;39;39#15;2;42;42;42#16;2;45;45;45#17;2;49;49;49#18;2;74;52;13#19;2;75;53;13#20;2;76;53;13#21;2;9;42;61#22;2;9;42;60#23;2;51;51;51#24;2;55;55;55#25;2;58;58;58#26;2;61;61;61#27;2;64;64;64#28;2;67;67;67#29;2;70;70;70#30;2;74;74;74#31;2;77;77;77#32;2;80;80;80#33;2;83;83;83#34;2;86;86;86#35;2;89;89;89#36;2;93;93;93#37;2;95;95;95#38;2;100;100;100#38!637~-!637~-!564~N^^N^^N^^N!4^N^^N!4^N!22~FFF!6~FFF!16~$#37!564?O??O??O??O!4?O??O!4?O!22?G?G!6?G?G$#10!564?_#6___??!13_?__#10_#33!22?O#0o!8?o#33O$#7!568?__!13?_#34!25?_#24G!8?G#34_$#35!611?o!6?o-#38!563~v#6~#21}!13~}!4~}{#6~!6?G#29C#27@@!7?@#34G???G!8?O?~$#37!563?G#22?@!13?@!4?@B#38?Z!4~B@w{{wx~B@w{{?@F~???~B@w{{???!16~$#37!587?c#15!4?G!7?_A???C!7?g#13Q?A#7A#6A$#31!592?C!7?C#8_??A!9?G#2A#26@#31?g$#11!592?O??A#36?D!7?@#30A!6?C?@?@$#14!592?_#0CA!

[]

In [9]:
perf.plot(CMP, "boxen", event="ipc", groupby="tier")


P0;0;0q"1;1;637;485#0;2;0;0;0#1;2;4;4;4#2;2;7;7;7#3;2;11;11;11#4;2;15;15;15#5;2;17;17;17#6;2;21;21;21#7;2;24;24;24#8;2;27;27;27#9;2;29;29;27#10;2;27;28;29#11;2;29;29;29#12;2;33;33;33#13;2;36;36;36#14;2;40;40;40#15;2;43;43;43#16;2;45;45;45#17;2;49;49;49#18;2;74;52;13#19;2;75;53;13#20;2;76;53;13#21;2;9;42;61#22;2;9;42;60#23;2;52;52;52#24;2;55;55;55#25;2;58;58;58#26;2;61;61;61#27;2;64;64;64#28;2;67;67;67#29;2;70;70;70#30;2;74;74;74#31;2;77;77;77#32;2;80;80;80#33;2;83;83;83#34;2;86;86;86#35;2;89;89;89#36;2;93;93;93#37;2;95;95;95#38;2;100;100;100#38!637~-!637~-!564~^^^N^^^N^^N^^^N^^N!4^N!22~FFF!6~FFF!16~$#8!564?_#6!18_?__#10_#37!22?G?G!6?G#33?_$#37!567?O???O??O???O??O!4?O#33!22?O#0o!8?o#34O$#7!583?_#34!25?_#24G!8?G#36G$#35!611?o!6?o-#38!563~u#6~#21}!18~}{#6~!6?G#1A!7?A??A#30A!6?C?@?@#6A$#37!563?H#22?@!18?@B#38?Z!4~B@w{{wx~B@w{{?@F~???~B@w{{???!16~$#37!587?c#14!4?_!7?_A???C!7?_#13Q?A#7A#31g$#11!592?O??A#27@!7?@#34G???_!8?O?~$#15!592?G#0C!7?C!4?w???~???C!4?~$#30!592?C#8_!7?_#35???_!5?~!6?@$

[]

## Two axes that did not work, and one that nearly did

**`itlb` was dropped on purpose.** Sweeping `itlb` hot vs cold on the same
function moved cycles only ~4% (528 vs 508) and moved the `fe-bound` share the
*wrong* way — 0.603 cold against 0.698 hot. Including it would have meant
shipping a claim the measurements do not support.

**A dependent chain cannot measure a front-end axis.** The first version of this
notebook used a single accumulator, so each add waited on the previous one. Cold
tiers showed a `fe-bound` share of 0.129 against 0.001 for hot — a 129x ratio
that looks like a dramatic result — while cycles stayed flat at 1115. Nothing had
been demonstrated. The share moved only because the slots re-attributed; no time
was spent. Eight independent accumulators fix it, and the `backend-bound` check
above now fails loudly if the body stops being backend-free.

**The checks assert which slot dominates, not a ratio.** `topdown-fe-bound`
under `hot` is small and noisy — it came out at 0.192, 0.200 and 0.296 across
runs, so a "cold is 3x hot" assertion passes or fails depending on the day. The
stable fact is that `cold` is frontend-bound by a wide margin and `hot` is not
bounded at all, so the checks above test slot dominance and the 0.5 threshold,
which both survive the noise.

**One real trap, shared with the retiring notebook.** A body of `x += constant`
folds to a single `add` at `-O2`, leaving nothing to fetch and nothing to
evict. `__asm__("" : "+r"(x))` keeps each accumulator live and the body real;
the cell above prints the resulting instruction count and code size so the
footprint is visible rather than assumed.